# Rodada 2: 4 novos datasets (kin8nm, news_popularity, santander, black_friday)

Antes de rodar: suba `runner.py`, `diagnostico.py` e `datasets_meta.py` (pasta `TCC_AutoML_Experimentos/common/`, **versões atualizadas**) para `/content/` pelo painel de arquivos do Colab.

Execução prevista: 4 ferramentas × 4 datasets × 5 sementes = **80 execuções** (≈ 7 h com orçamento de 300 s; o Colab gratuito pode cair, então rode de novo a célula 5 quantas vezes precisar). O runner pula o que já existe no Drive. Os resultados vão para a **mesma pasta** `TCC_resultados` das 158 execuções anteriores (arquivos novos, nada é sobrescrito). Use GPU T4 (Ambiente de execução > Alterar tipo > T4).

In [ ]:
# 1) Mesmas versões das rodadas anteriores (não misturar versões)
!pip install -q --default-timeout=200 autogluon.tabular==1.5.0 catboost==1.2.10 flaml==2.7.0 h2o==3.46.0.12 tpot==1.1.0
# Se o Colab pedir, reinicie a sessão (Ambiente de execução > Reiniciar sessão) e rode daqui para baixo.

In [ ]:
# 2) Confere se os 3 arquivos estão em /content
import sys, os, glob, collections
sys.path.append("/content")
faltam = [f for f in ["runner.py", "diagnostico.py", "datasets_meta.py"] if not os.path.exists(f"/content/{f}")]
print("Arquivos faltando em /content:", faltam or "nenhum")
import datasets_meta
print("Datasets novos no catálogo:", [d for d in ["kin8nm", "news_popularity", "santander", "black_friday"] if d in datasets_meta.CANDIDATOS])

In [ ]:
# 3) Drive + registro do ambiente desta sessão em arquivo separado
from google.colab import drive
drive.mount("/content/drive")
DIR = "/content/drive/MyDrive/TCC_resultados"
os.makedirs(DIR, exist_ok=True)
from diagnostico import registrar
registrar(f"{DIR}/ambiente_rodada2.json")
# ATENÇÃO: confira se gpu_disponivel é True (T4).

In [ ]:
# 3b) (opcional, ~2 min) Metadados dos 4 novos datasets: confira linhas/atributos antes de rodar
!cd /content && python datasets_meta.py kin8nm news_popularity santander black_friday
# Esperado: kin8nm 8192x8 | news_popularity 39644x59 | santander 4459x4991 | black_friday 166821x9

In [ ]:
# 4b) DIAGNÓSTICO: tempo e resultado de cada execução já feita + falhas registradas (cole a saída no chat)
import json, glob, pandas as pd
rows = []
for d in NOVOS:
    for f in sorted(glob.glob(f"{DIR}/*__{d}__seed*.json")):
        r = json.load(open(f, encoding="utf-8"))
        rows.append((r["framework"], r["dataset"], r["seed"], r["tempo_real_s"], round(r["ram_pico_mb"]), round(r["ram_delta_mb"]), r["rmse"]))
df = pd.DataFrame(rows, columns=["ferramenta", "dataset", "seed", "tempo_s", "ram_pico_mb", "ram_delta_mb", "rmse"])
pd.set_option("display.width", 200)
print(df.to_string(index=False))
print("
Tempo total das execuções concluídas: %.1f h" % (df.tempo_s.sum() / 3600))
print(df.groupby(["dataset", "ferramenta"]).tempo_s.agg(["count", "mean", "max"]).round(0))
for f in sorted(glob.glob(f"{DIR}/*.falha")):
    print("FALHA:", json.load(open(f, encoding="utf-8")))

In [ ]:
# 4) Progresso atual no Drive (cada semente completa = 16 execuções novas)
NOVOS = ["kin8nm", "news_popularity", "santander", "black_friday"]
c = collections.Counter()
for d in NOVOS:
    for f in glob.glob(f"{DIR}/*__{d}__seed*.json"):
        c[f.split("seed")[1].split(".")[0]] += 1
for s in range(42, 47):
    print(f"seed {s}: {c.get(str(s), 0)}/16")

In [ ]:
# 5) Roda o lote (semente por semente; pula o que já existe). Pode rodar de novo se o Colab cair.
import runner
print("TPOT_PARAMS:", runner.TPOT_PARAMS)  # deve ser {'cv': 3, 'max_eval_time_mins': 1}
TOOLS = ["flaml", "autogluon", "h2o", "tpot"]
# cada execução roda em processo isolado (memória zerada a cada vez) e com limite rígido de 1200 s; falhas ficam em arquivos .falha
runner.executar_tudo(TOOLS, NOVOS, seeds=range(42, 47), budget=300, dir_=DIR)

In [ ]:
# 6) Resumo: o que ficou pronto e o que falhou (envie esta saída para análise)
faltam = [(t, d, s) for t in TOOLS for d in NOVOS for s in range(42, 47)
          if not os.path.exists(f"{DIR}/{t}__{d}__seed{s}.json")]
print(f"Concluídas: {80 - len(faltam)}/80")
for t, d, s in faltam:
    print("  falta:", t, d, s)